# 04. Backtest Analysis
**Objective:** Analyze the performance of the strategy simulation.

**Requires:** `results/backtest_mean_variance/equity_curve.csv` (generated by `run_backtest.py` or pipeline).

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path('../').resolve()
sys.path.append(str(PROJECT_ROOT))

from config.settings import config
from quant_alpha.visualization import plot_equity_curve, plot_drawdown, plot_monthly_heatmap
from quant_alpha.utils import calculate_returns

%matplotlib inline

## 1. Load Equity Curve

In [ ]:
eq_path = config.RESULTS_DIR / 'backtest_mean_variance' / 'equity_curve.csv'

if not eq_path.exists():
    raise FileNotFoundError("Backtest results not found. Run scripts/run_backtest.py first.")

equity = pd.read_csv(eq_path)
equity['date'] = pd.to_datetime(equity['date'])
equity = equity.set_index('date')

print(f"Loaded backtest: {equity.index.min().date()} -> {equity.index.max().date()}")

## 2. Performance Metrics

In [ ]:
returns = equity['total_value'].pct_change().dropna()
cagr = (1 + returns).prod() ** (252 / len(returns)) - 1
sharpe = returns.mean() / returns.std() * (252 ** 0.5)
max_dd = (equity['total_value'] / equity['total_value'].cummax() - 1).min()

print(f"CAGR:      {cagr:.2%}")
print(f"Sharpe:    {sharpe:.2f}")
print(f"Max DD:    {max_dd:.2%}")

## 3. Visualizations

In [ ]:
plot_equity_curve(equity.reset_index())

In [ ]:
plot_drawdown(equity.reset_index())

In [ ]:
plot_monthly_heatmap(returns)